# File Hashing & Evidence Integrity
**Module: Month 01 — Digital Forensics Foundations**  
**Date: 2026-03-22**

---

## Concept
In forensic investigations, **hash verification** is the chain-of-custody mechanism for digital evidence.  
A cryptographic hash is a deterministic fingerprint: the same input always produces the same output, and any change — even a single bit — produces a completely different output (the **avalanche effect**).

**MD5** (128-bit): Fast, widely used in legacy forensics tools. Considered cryptographically broken for security, but still valid for integrity checking.

**SHA-256** (256-bit): Current standard. Used by Autopsy, FTK, and modern DFIR workflows. Collision-resistant.

> **Rule**: Hash before you touch. Hash after you touch. If they match — you didn't tamper with the evidence.

---
## 1. Imports

In [1]:
import hashlib
import os
from pathlib import Path
from datetime import datetime

---
## 2. Core Hashing Function

We read the file in **chunks** (not all at once) to handle large evidence files without blowing up RAM. This is how real forensic tools do it.

In [2]:
def hash_file(filepath: str, chunk_size: int = 65536) -> dict:
    """
    Compute MD5 and SHA-256 hashes of a file.
    Reads in chunks to support large files.

    Parameters
    ----------
    filepath  : path to the target file
    chunk_size: bytes per read (default 64 KB)

    Returns
    -------
    dict with keys: filepath, size_bytes, md5, sha256, timestamp
    """
    path = Path(filepath)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {filepath}")
    if not path.is_file():
        raise ValueError(f"Not a file: {filepath}")

    md5    = hashlib.md5()
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            md5.update(chunk)
            sha256.update(chunk)

    return {
        "filepath"   : str(path.resolve()),
        "filename"   : path.name,
        "size_bytes" : path.stat().st_size,
        "md5"        : md5.hexdigest(),
        "sha256"     : sha256.hexdigest(),
        "timestamp"  : datetime.utcnow().isoformat() + "Z"
    }


def print_hash_report(result: dict) -> None:
    """Pretty-print a hash_file() result."""
    print(f"  File     : {result['filename']}")
    print(f"  Path     : {result['filepath']}")
    print(f"  Size     : {result['size_bytes']} bytes")
    print(f"  MD5      : {result['md5']}")
    print(f"  SHA-256  : {result['sha256']}")
    print(f"  Hashed at: {result['timestamp']}")
    print()

---
## 3. Create 3 Test Evidence Files

In [3]:
# Create a scratch directory for test files
evidence_dir = Path("./evidence_test")
evidence_dir.mkdir(exist_ok=True)

# File 1: a plain text document
file1 = evidence_dir / "report.txt"
file1.write_text("Suspect accessed the server at 03:42 UTC. No other activity logged.")

# File 2: a short CSV (simulating a log export)
file2 = evidence_dir / "access_log.csv"
file2.write_text("timestamp,user,action\n2026-03-10T03:42:00Z,jdoe,LOGIN\n2026-03-10T04:01:00Z,jdoe,LOGOUT\n")

# File 3: a binary-ish file (simulating metadata)
file3 = evidence_dir / "metadata.bin"
file3.write_bytes(bytes(range(256)) * 4)  # 1024 bytes of 0x00-0xFF repeated

print("Test files created:")
for f in sorted(evidence_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

Test files created:
  access_log.csv  (90 bytes)
  metadata.bin  (1024 bytes)
  report.txt  (67 bytes)


---
## 4. Hash All Three Files — Baseline

In [4]:
files = [file1, file2, file3]
baseline = {}

print("=" * 60)
print("BASELINE HASHES")
print("=" * 60)

for f in files:
    result = hash_file(f)
    baseline[f.name] = result
    print_hash_report(result)

BASELINE HASHES
  File     : report.txt
  Path     : C:\Users\nblenovo\Documents\Projects\Forensics studies\evidence_test\report.txt
  Size     : 67 bytes
  MD5      : 424c657c3699cc34d17a42ca1c8f4297
  SHA-256  : 0a7a48377b92ca02d7a82e687b7c0be0b778c365631f35e98415a46e339c1f18
  Hashed at: 2026-03-22T09:48:04.782919Z

  File     : access_log.csv
  Path     : C:\Users\nblenovo\Documents\Projects\Forensics studies\evidence_test\access_log.csv
  Size     : 90 bytes
  MD5      : 1fd1741f1deff0fa5f1ee09ed7be4291
  SHA-256  : e2aecba040370119d79c02dfe23ea86a5f72e0d26add5e77e121771bdde4398d
  Hashed at: 2026-03-22T09:48:04.790804Z

  File     : metadata.bin
  Path     : C:\Users\nblenovo\Documents\Projects\Forensics studies\evidence_test\metadata.bin
  Size     : 1024 bytes
  MD5      : b2ea9f7fcea831a4a63b213f41a8855b
  SHA-256  : 785b0751fc2c53dc14a4ce3d800e69ef9ce1009eb327ccf458afe09c242c26c9
  Hashed at: 2026-03-22T09:48:04.797875Z



---
## 5. Tamper with One File — Add a Single Character

We append a single space to `report.txt`. In a real investigation, this is the equivalent of someone making the tiniest edit imaginable to an evidence file — almost invisible visually, but **forensically catastrophic**.

In [5]:
# Append a single space character to report.txt
with open(file1, "a") as f:
    f.write(" ")   # <-- one trailing space

print(f"report.txt modified: appended a single space character.")
print(f"New size: {file1.stat().st_size} bytes")

report.txt modified: appended a single space character.
New size: 68 bytes


---
## 6. Re-hash All Files & Compare Against Baseline

In [6]:
print("=" * 60)
print("POST-TAMPER INTEGRITY CHECK")
print("=" * 60)

for f in files:
    current = hash_file(f)
    original = baseline[f.name]

    md5_match    = current["md5"]    == original["md5"]
    sha256_match = current["sha256"] == original["sha256"]
    status = "✅ INTACT" if (md5_match and sha256_match) else "🚨 TAMPERED"

    print(f"--- {f.name} ---  {status}")
    if not md5_match:
        print(f"  MD5  (original): {original['md5']}")
        print(f"  MD5  (current) : {current['md5']}")
    if not sha256_match:
        print(f"  SHA256 (original): {original['sha256']}")
        print(f"  SHA256 (current) : {current['sha256']}")
    print()

POST-TAMPER INTEGRITY CHECK
--- report.txt ---  🚨 TAMPERED
  MD5  (original): 424c657c3699cc34d17a42ca1c8f4297
  MD5  (current) : 8e7cc6f84831b1da2d0f274789c751a8
  SHA256 (original): 0a7a48377b92ca02d7a82e687b7c0be0b778c365631f35e98415a46e339c1f18
  SHA256 (current) : 03cea0eff42380b6ebc63642a6b79d62b60f9c3bbba7b32f3406d96a43f5b7ea

--- access_log.csv ---  ✅ INTACT

--- metadata.bin ---  ✅ INTACT



---
## 7. The Avalanche Effect — Visualized

Let's make the avalanche effect explicit by hashing nearly-identical strings and counting how many bits differ.

In [7]:
def count_bit_differences(hex1: str, hex2: str) -> int:
    """Count the number of differing bits between two hex-encoded hashes."""
    int1 = int(hex1, 16)
    int2 = int(hex2, 16)
    xor  = int1 ^ int2
    return bin(xor).count("1")


original_text  = b"Suspect accessed the server at 03:42 UTC. No other activity logged."
tampered_text  = b"Suspect accessed the server at 03:42 UTC. No other activity logged. "

hash_orig = hashlib.sha256(original_text).hexdigest()
hash_tamp = hashlib.sha256(tampered_text).hexdigest()

diff_bits = count_bit_differences(hash_orig, hash_tamp)
total_bits = 256

print("AVALANCHE EFFECT DEMO (SHA-256)")
print(f"  Input change  : 1 trailing space appended")
print(f"  Original hash : {hash_orig}")
print(f"  Tampered hash : {hash_tamp}")
print(f"  Bits changed  : {diff_bits} / {total_bits}  ({diff_bits/total_bits*100:.1f}% of the hash flipped)")
print()
print("Ideal avalanche = ~50% of bits flip. A hash that doesn't do this is weak.")

AVALANCHE EFFECT DEMO (SHA-256)
  Input change  : 1 trailing space appended
  Original hash : 0a7a48377b92ca02d7a82e687b7c0be0b778c365631f35e98415a46e339c1f18
  Tampered hash : 03cea0eff42380b6ebc63642a6b79d62b60f9c3bbba7b32f3406d96a43f5b7ea
  Bits changed  : 122 / 256  (47.7% of the hash flipped)

Ideal avalanche = ~50% of bits flip. A hash that doesn't do this is weak.


---
## 8. Forensic Hash Log — Export to File

In real casework, you'd write hashes to a **manifest file** that gets signed and timestamped — the chain of custody document.

In [8]:
import json

manifest_path = evidence_dir / "hash_manifest.json"

manifest = {
    "case_id"     : "CASE-2026-001",
    "examiner"    : "Anas",
    "generated_at": datetime.utcnow().isoformat() + "Z",
    "files"       : list(baseline.values())
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Hash manifest written to: {manifest_path.resolve()}")
print()
print(json.dumps(manifest, indent=2))

Hash manifest written to: C:\Users\nblenovo\Documents\Projects\Forensics studies\evidence_test\hash_manifest.json

{
  "case_id": "CASE-2026-001",
  "examiner": "Anas",
  "generated_at": "2026-03-22T09:49:21.006644Z",
  "files": [
    {
      "filepath": "C:\\Users\\nblenovo\\Documents\\Projects\\Forensics studies\\evidence_test\\report.txt",
      "filename": "report.txt",
      "size_bytes": 67,
      "md5": "424c657c3699cc34d17a42ca1c8f4297",
      "sha256": "0a7a48377b92ca02d7a82e687b7c0be0b778c365631f35e98415a46e339c1f18",
      "timestamp": "2026-03-22T09:48:04.782919Z"
    },
    {
      "filepath": "C:\\Users\\nblenovo\\Documents\\Projects\\Forensics studies\\evidence_test\\access_log.csv",
      "filename": "access_log.csv",
      "size_bytes": 90,
      "md5": "1fd1741f1deff0fa5f1ee09ed7be4291",
      "sha256": "e2aecba040370119d79c02dfe23ea86a5f72e0d26add5e77e121771bdde4398d",
      "timestamp": "2026-03-22T09:48:04.790804Z"
    },
    {
      "filepath": "C:\\Users\\nblenov

---
## 9. Key Takeaways

| Observation | Forensic meaning |
|---|---|
| A 1-character change flips ~50% of hash bits | Avalanche effect — no partial matches, no "close enough" |
| MD5 and SHA-256 both change completely | Redundant hashing = stronger chain of custody |
| File size changed by 1 byte | Metadata alone can betray tampering, but hash is definitive |
| Untouched files have identical hashes | Hash is stable — same input, same output, always |

**MD5 vs SHA-256 in practice:**  
MD5 is no longer collision-resistant (two *different* inputs can produce the same MD5), but finding such collisions takes deliberate effort. For simple integrity checking it's still used. SHA-256 has no known collisions and is the current DFIR standard. Always compute both — Autopsy, FTK Imager, and X-Ways all do.

> **The investigator's rule**: A file's hash is its fingerprint. Fingerprints don't lie — but they need to be taken at the right moment.

---
## 10. Cleanup (optional)

In [9]:
# Run this cell only if you want to delete the test files
import shutil
# shutil.rmtree(evidence_dir)
# print("Test directory removed.")
print("Cleanup skipped — test files preserved in ./evidence_test/")

Cleanup skipped — test files preserved in ./evidence_test/
